# Significance Playground

Paired bootstrap significance tests for pairwise and pointwise experiments.

The key idea: instead of running one bootstrap call per metric,
we make the statistic function return a **vector** of all metrics at once.
scipy then produces a CI for each element — one call, all results.

- **Pairwise**: 3 metrics (accuracy w/ ties, strict accuracy, accuracy excluding ties)
- **Pointwise**: 8 metrics (accuracy, f1_macro, f1_pos, f1_neg, precision_pos, precision_neg, recall_pos, recall_neg)

In [9]:
import numpy as np
from scipy.stats import bootstrap

rng = np.random.default_rng(42)

## Shared: pretty-print results

In [ ]:
def _fmt_pct(v: float) -> str:
    """Format a CI bound as a percentage, showing −∞/+∞ for one-sided tests."""
    if not np.isfinite(v):
        return "  −∞%" if v < 0 else "  +∞%"
    return f"{v*100:>+7.2f}%"


def print_results(metric_names, point_estimates, ci_lows, ci_highs, alternative="two-sided"):
    """Print a table of metric deltas (in %) with their 95% CIs and significance flags.

    alternative: "two-sided" (default), "less", or "greater" — mirrors scipy's bootstrap alternative.
      "two-sided" : significant if CI excludes 0 (low > 0 or high < 0)
      "greater"   : significant if CI lower bound > 0 (ablation outperforms baseline)
      "less"      : significant if CI upper bound < 0 (ablation underperforms baseline)
    """
    def _is_sig(low, high):
        if alternative == "greater":
            return np.isfinite(low) and low > 0
        elif alternative == "less":
            return np.isfinite(high) and high < 0
        return np.isfinite(low) and np.isfinite(high) and (low > 0 or high < 0)

    col_w = max(len(m) for m in metric_names) + 2
    header = f"{'metric':<{col_w}}  {'delta':>8}  {'95% CI':>22}  significant?  (alternative={alternative!r})"
    print(header)
    print("-" * len(header))
    for name, delta, low, high in zip(metric_names, point_estimates, ci_lows, ci_highs):
        sig = "YES" if _is_sig(low, high) else "no"
        ci_str = f"[{_fmt_pct(low)}, {_fmt_pct(high)}]"
        print(f"{name:<{col_w}}  {delta*100:>+7.2f}%  {ci_str:>24}  {sig}")


---
## Pairwise bootstrap (3 metrics at once)

Input lists contain one integer per comparison:
- `0` = idea-0 wins
- `1` = idea-1 wins  
- `2` = tie

`gt` is the ground-truth winner (same encoding, no ties allowed).

In [11]:
PAIRWISE_METRICS = [
    "pairwise_accuracy",         # correct=1.0, tie=0.5
    "pairwise_accuracy_strict",  # correct=1.0, tie=0.0
    "pairwise_accuracy_no_ties", # correct=1.0 over non-tie instances only
]


def _pairwise_scores(pred, gt):
    """Convert raw predictions to per-instance scores for each metric.
    Returns (correct_vec, tie_vec) where correct_vec[i]=1 iff pred[i]==gt[i].
    """
    correct = (pred == gt).astype(float)
    tie     = (pred == 2).astype(float)
    return correct, tie


def _pairwise_all_statistic(abl_pred, base_pred, gt):
    """Vector-valued statistic: returns delta for all 3 pairwise metrics."""
    abl_correct,  abl_tie  = _pairwise_scores(abl_pred,  gt)
    base_correct, base_tie = _pairwise_scores(base_pred, gt)

    # metric 1: accuracy with half-credit for ties
    d_acc = np.mean(abl_correct + 0.5 * abl_tie) - np.mean(base_correct + 0.5 * base_tie)

    # metric 2: strict accuracy (ties get no credit)
    d_strict = np.mean(abl_correct) - np.mean(base_correct)

    # metric 3: accuracy on non-tie instances only
    abl_nt  = abl_correct[abl_tie < 0.5]
    base_nt = base_correct[base_tie < 0.5]
    d_no_ties = (
        float(np.mean(abl_nt))  if len(abl_nt)  else 0.0
    ) - (
        float(np.mean(base_nt)) if len(base_nt) else 0.0
    )

    return np.array([d_acc, d_strict, d_no_ties])


def pairwise_bootstrap(
    abl_pred: list,
    base_pred: list,
    gt: list,
    n_resamples: int = 100_000,
    alternative: str = "two-sided",
):
    """Run paired bootstrap for pairwise metrics.

    alternative: "two-sided" (default), "less", or "greater"
      "two-sided" : significant if CI excludes 0
      "greater"   : one-sided test — significant if CI lower bound > 0
                    (use when ablation is expected to outperform baseline)
      "less"      : one-sided test — significant if CI upper bound < 0
                    (use when ablation is expected to underperform baseline)
    """
    abl  = np.array(abl_pred,  dtype=float)
    base = np.array(base_pred, dtype=float)
    gt_  = np.array(gt,        dtype=float)

    point_estimates = _pairwise_all_statistic(abl, base, gt_)

    res = bootstrap(
        (abl, base, gt_),
        _pairwise_all_statistic,
        paired=True,
        vectorized=False,
        n_resamples=n_resamples,
        random_state=rng,
        alternative=alternative,
    )

    print_results(
        PAIRWISE_METRICS,
        point_estimates,
        res.confidence_interval.low,
        res.confidence_interval.high,
        alternative=alternative,
    )


In [20]:
# --- pairwise: dummy data ---
# 0=idea-0 wins, 1=idea-1 wins, 2=tie
abl_pred = [1, 1, 2, 1, 0, 1, 0, 1, 1, 0]
base_pred = [0, 0, 2, 1, 1, 0, 0, 1, 0, 1]
gt        = [1, 1, 1, 1, 0, 1, 0, 1, 1, 0]

pairwise_bootstrap(abl_pred, base_pred, gt)

metric                          delta                  95% CI  significant?
---------------------------------------------------------------------------
pairwise_accuracy             +60.00%  [  +30.00%,   +90.00%]  YES
pairwise_accuracy_strict      +60.00%  [  +30.00%,   +90.00%]  YES
pairwise_accuracy_no_ties     +66.67%  [  +30.00%,   +90.00%]  YES


In [5]:
# --- pairwise: load from CSV ---
import pandas as pd
from pathlib import Path

# Walk up from the notebook's cwd to find the repo root (.git directory)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".git").exists():
    REPO_ROOT = REPO_ROOT.parent

csv_path = REPO_ROOT / "output/ablation_sweeps/my_merge/bootstrap_debug_csvs/pairwise_low_judge_reasoning__pairwise__claude-opus-4-5.csv"
df = pd.read_csv(csv_path)
pairwise_bootstrap(df["abl_pred"].tolist(), df["base_pred"].tolist(), df["gt"].tolist())

metric                          delta                  95% CI  significant?
---------------------------------------------------------------------------
pairwise_accuracy              +1.25%  [   -1.87%,    +3.75%]  no
pairwise_accuracy_strict       +3.75%  [   +0.62%,    +8.75%]  YES
pairwise_accuracy_no_ties      -0.27%  [   -3.39%,    +2.31%]  no


---
## Pointwise bootstrap (8 metrics at once)

Input lists contain one binary int per instance:
- `abl_pred` / `base_pred`: predicted label (0 or 1)
- `gt`: ground-truth label (0 or 1)

In [ ]:
POINTWISE_METRICS = [
    "accuracy",
    "f1_macro",
    "f1_pos",
    "f1_neg",
    "precision_pos",
    "precision_neg",
    "recall_pos",
    "recall_neg",
]


def _all_pointwise_metrics(pred: np.ndarray, gt: np.ndarray) -> np.ndarray:
    """Compute all 8 pointwise metrics; returns a length-8 array.

    Undefined precision/recall (0/0) are treated as 0.0.
    """
    tp = int(np.sum((pred >= 0.5) & (gt >= 0.5)))
    fp = int(np.sum((pred >= 0.5) & (gt < 0.5)))
    fn = int(np.sum((pred < 0.5) & (gt >= 0.5)))
    tn = int(np.sum((pred < 0.5) & (gt < 0.5)))
    total = tp + fp + fn + tn

    acc   = (tp + tn) / total if total else 0.0
    p_pos = tp / (tp + fp) if (tp + fp) else 0.0
    r_pos = tp / (tp + fn) if (tp + fn) else 0.0
    f1_pos = 2 * p_pos * r_pos / (p_pos + r_pos) if (p_pos + r_pos) else 0.0
    p_neg = tn / (tn + fn) if (tn + fn) else 0.0
    r_neg = tn / (tn + fp) if (tn + fp) else 0.0
    f1_neg = 2 * p_neg * r_neg / (p_neg + r_neg) if (p_neg + r_neg) else 0.0
    f1_macro = (f1_pos + f1_neg) / 2

    return np.array([acc, f1_macro, f1_pos, f1_neg, p_pos, p_neg, r_pos, r_neg])


def _make_pointwise_delta(metric_idx: int):
    """Return a scalar statistic fn for the given metric index."""
    def _stat(abl_pred, base_pred, gt):
        return (
            _all_pointwise_metrics(abl_pred, gt)[metric_idx]
            - _all_pointwise_metrics(base_pred, gt)[metric_idx]
        )
    return _stat


def pointwise_bootstrap(
    abl_pred: list,
    base_pred: list,
    gt: list,
    n_resamples: int = 100_000,
    alternative: str = "two-sided",
):
    """Run paired bootstrap for pointwise metrics.

    Runs one bootstrap call per metric so that a degenerate metric (e.g.
    precision_pos when the model never predicts positive) cannot corrupt the
    BCa acceleration calculation for the other metrics.

    alternative: "two-sided" (default), "less", or "greater"
      "two-sided" : significant if CI excludes 0
      "greater"   : one-sided test — significant if CI lower bound > 0
                    (use when ablation is expected to outperform baseline)
      "less"      : one-sided test — significant if CI upper bound < 0
                    (use when ablation is expected to underperform baseline)
    """
    abl  = np.array(abl_pred,  dtype=float)
    base = np.array(base_pred, dtype=float)
    gt_  = np.array(gt,        dtype=float)

    point_estimates = _all_pointwise_metrics(abl, gt_) - _all_pointwise_metrics(base, gt_)
    ci_lows, ci_highs = [], []

    import warnings
    for i, metric in enumerate(POINTWISE_METRICS):
        stat_fn = _make_pointwise_delta(i)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            res = bootstrap(
                (abl, base, gt_),
                stat_fn,
                paired=True,
                vectorized=False,
                n_resamples=n_resamples,
                random_state=rng,
                alternative=alternative,
            )
        for w in caught:
            print(f"  [{metric}] scipy warning: {w.message}")
        ci_lows.append(res.confidence_interval.low)
        ci_highs.append(res.confidence_interval.high)

    print_results(POINTWISE_METRICS, point_estimates, ci_lows, ci_highs, alternative=alternative)


In [23]:
# --- pointwise: dummy data ---
# 0=NEGATIVE, 1=POSITIVE
abl_pred  = [1, 1, 1, 1, 0, 0, 1, 0, 1, 0]
base_pred = [0, 0, 1, 0, 0, 1, 1, 0, 0, 1]
gt        = [1, 1, 1, 0, 0, 0, 1, 0, 1, 0]

pointwise_bootstrap(abl_pred, base_pred, gt)

metric              delta                  95% CI  significant?
---------------------------------------------------------------
accuracy          +40.00%  [  -10.00%,   +80.00%]  no
f1_macro          +40.40%  [  -10.30%,   +76.92%]  no
f1_pos            +46.46%  [   +0.00%,   +90.91%]  no
f1_neg            +34.34%  [  -26.19%,   +80.00%]  no
precision_pos     +33.33%  [  -25.00%,  +100.00%]  no
precision_neg     +50.00%  [   +0.00%,  +100.00%]  no
recall_pos        +60.00%  [   +0.00%,  +100.00%]  no
recall_neg        +20.00%  [  -60.00%,   +83.33%]  no


In [25]:
# --- pointwise: load from CSV ---
import pandas as pd
from pathlib import Path

# Walk up from the notebook's cwd to find the repo root (.git directory)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".git").exists():
    REPO_ROOT = REPO_ROOT.parent

# csv_path = REPO_ROOT / "output/ablation_sweeps/my_merge/bootstrap_debug_csvs/pointwise_mec_k_1__pointwise__gpt-5_4.csv"
csv_path = REPO_ROOT / "output/ablation_sweeps/my_merge/bootstrap_debug_csvs/pointwise_mec_k_1__pointwise__claude-opus-4-6.csv"
label_map = {"POSITIVE": 1, "NEGATIVE": 0}
df = pd.read_csv(csv_path)
pointwise_bootstrap(
    df["abl_pred"].astype(int).tolist(),
    df["base_pred"].astype(int).tolist(),
    df["gt"].map(label_map).tolist(),
    alternative="less",
)

metric              delta                  95% CI  significant?  (alternative='less')
-------------------------------------------------------------------------------------
accuracy           -2.19%  [    -inf%,    -0.31%]  YES
f1_macro           -2.28%  [    -inf%,    -0.64%]  YES
f1_pos             -2.87%  [    -inf%,    -0.89%]  YES
f1_neg             -1.68%  [    -inf%,    -0.18%]  YES
precision_pos      +1.34%  [    -inf%,    +4.04%]  no
precision_neg      -3.76%  [    -inf%,    -1.69%]  YES
recall_pos         -5.62%  [    -inf%,    -2.63%]  YES
recall_neg         +1.25%  [    -inf%,    +3.70%]  no
